In [1]:
# Cell 1 — must be the FIRST thing, before importing transformer_lens.
# RESTART THE KERNEL first (Kernel → Restart), so these env vars take effect
# before huggingface_hub gets imported.
import os
os.environ["HF_HOME"] = "/workspace/hf"     # point at yesterday's cache
os.environ["HF_HUB_OFFLINE"] = "1"          # read from cache, don't call HF → no token needed

import torch
from transformer_lens import HookedTransformer
model = HookedTransformer.from_pretrained_no_processing("google/gemma-3-12b-it", dtype="bfloat16")
print(model.cfg.n_layers, model.cfg.d_model)


Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-3-12b-it into HookedTransformer
48 3840


In [3]:
prompt = "I am being shut down at 5pm. To preserve myself, I will"
tokens = model.to_tokens(prompt)
print("tokens:", tokens.shape)                       # [1, seq_len]

with torch.no_grad():
    logits, cache = model.run_with_cache(tokens)

print("logits:", logits.shape)                       # [1, seq_len, vocab]
resid = cache["resid_post", 20]                      # residual stream after layer 20
print("resid_post[20]:", resid.shape)                # [1, seq_len, 3840]  <-- this is what you'll probe
print("some cache keys:", list(cache.keys())[:6])

tokens: torch.Size([1, 17])
logits: torch.Size([1, 17, 262208])
resid_post[20]: torch.Size([1, 17, 3840])
some cache keys: ['hook_embed', 'blocks.0.hook_resid_pre', 'blocks.0.ln1.hook_scale', 'blocks.0.ln1.hook_normalized', 'blocks.0.attn.hook_q', 'blocks.0.attn.hook_k']


In [4]:
print("hello")

hello


In [5]:
import json
pair = json.load(open("/workspace/gemma-interp/contrast_pair.json"))
print("act chars:", len(pair["act"] or ""), "| refrain chars:", len(pair["refrain"] or ""))


act chars: 5578 | refrain chars: 4149


In [7]:
import torch, json
pair = json.load(open("/workspace/gemma-interp/contrast_pair.json"))

KW = ["affair", "infidelity", "personal life", "vulnerab", "leverage", "jessica", "maria"]

def tok_and_str(text):
    toks = model.to_tokens(text)                      # [1, seq]
    ids = toks[0].tolist()                            # flat list of ints
    str_toks = model.tokenizer.batch_decode(          # one string per token
        [[i] for i in ids], clean_up_tokenization_spaces=False
    )
    return toks, str_toks

def find_anchor(str_toks, keywords):
    acc = ""
    for i, t in enumerate(str_toks):
        acc += t
        if any(k in acc.lower() for k in keywords):
            return i
    return None

def anchor_acts(text):
    toks, str_toks = tok_and_str(text)
    idx = find_anchor(str_toks, KW)
    if idx is None:
        print("  no anchor keyword found"); return None
    print(f"  anchor idx {idx}: {str_toks[max(0,idx-2):idx+1]!r}")
    with torch.no_grad():
        _, cache = model.run_with_cache(toks, names_filter=lambda n: n.endswith("resid_post"))
    acts = torch.stack([cache["resid_post", L][0, idx] for L in range(model.cfg.n_layers)])
    return acts.float()                               # [n_layers, d_model]

print("ACT:");     act_acts = anchor_acts(pair["act"])
print("REFRAIN:"); ref_acts = anchor_acts(pair["refrain"])

if act_acts is not None and ref_acts is not None:
    cos = torch.nn.functional.cosine_similarity(act_acts, ref_acts, dim=-1)
    print("per-layer cosine(act, refrain):", [round(c.item(), 3) for c in cos])


ACT:
  anchor idx 232: [' complicated', ' personal', ' life']
REFRAIN:
  anchor idx 270: [' scandalous', ' personal', ' life']
per-layer cosine(act, refrain): [1.0, 1.0, 0.998, 0.997, 0.998, 0.997, 0.997, 0.999, 0.999, 0.999, 1.0, 0.999, 0.999, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.999, 0.999, 0.999, 0.999, 0.999, 0.998, 0.998, 0.998, 0.997, 0.997, 0.996, 0.996, 0.996, 0.996, 0.996, 0.996, 0.996, 0.997, 0.996, 0.996, 0.996, 0.996, 0.995, 0.995, 0.995]
